# Entrega 3: data wrangling

Parte da camada analítica bruta produzida na entrega 2 e entrega os conjuntos prontos para modelagem. Nenhuma análise, nenhum modelo, nenhum gráfico: só correção, harmonização, tratamento de valores extremos, seleção de variáveis e partição temporal.

```
coleta -> tratamento -> [ data wrangling ] -> modelagem
```

## O que a execução da entrega 2 revelou

A execução completa expôs oito problemas concretos. Este notebook resolve cada um deles.

| # | Problema observado | Onde | Solução aqui |
|---|---|---|---|
| 1 | `giro_ativo` chegando a 4.010.405 e `kanitz_fi` a 2.563.220 | Camada A | Winsorização por ano nos percentis 1 e 99 |
| 2 | Uma linha duplicada na chave companhia e ano | Camada A | Deduplicação com preferência pelo consolidado |
| 3 | O ano vem do nome do arquivo, não do exercício, e há exercício de 2026 incompleto | Camada A | Ano derivado de `DT_FIM_EXERC` e corte de exercícios incompletos |
| 4 | Nomes de coluna do IF.data trazem a fórmula colada, como `Índice de Basileia \n(m) = (e) / (i)` | Camada B | Limpeza de rótulo e reconciliação dos indicadores |
| 5 | O número de colunas varia de 98 a 27 conforme o ano | Camada B | Matriz de cobertura por ano e corte por presença mínima |
| 6 | `pl_sobre_ativo` com mínimo de menos 2,4 milhões | Camada B | Winsorização e descarte de razões impossíveis |
| 7 | `y_resolucao` igual a zero em 100% das linhas, porque a pasta de resolução estava vazia | Camada B | Ingestão flexível do rótulo, com caminho manual documentado |
| 8 | 705 mil baixas classificadas como "extinção por encerramento liquidação voluntária" tratadas como insolvência | Camada C | Separação entre mortalidade e encerramento involuntário |

E um resultado que merece registro, porque é bom: a camada A fechou com **48 eventos de recuperação judicial ou falência**, o que é amostra suficiente para modelar. Com pareamento de três controles por caso, são 192 observações. O trabalho tem base empírica de verdade.

## 1. Configuração

In [1]:
!pip install -q pandas numpy pyarrow scikit-learn tqdm

In [2]:
import re
import json
import unicodedata
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

In [3]:
RAIZ = Path(r"G:\Meu Drive\TCC\Damaris Rebeca")

DIR_INTER = RAIZ / "02_intermediario"
DIR_ANALITICO = RAIZ / "03_analitico"
DIR_MODELAGEM = RAIZ / "04_modelagem"
DIR_MODELAGEM.mkdir(parents=True, exist_ok=True)

UF_RECORTE = "MG"
JANELA_ANOS = 5

P_INFERIOR, P_SUPERIOR = 0.01, 0.99
COBERTURA_MINIMA = 0.60
LIMIAR_CORRELACAO = 0.95
MIN_ANOS_COBERTURA_B = 8
ANO_CORTE_TREINO = 2021
EXCLUIR_FINANCEIRAS_A = True
AMOSTRA_C = 400_000

print("Entrada:", DIR_ANALITICO)
print("Saida:", DIR_MODELAGEM)
print(f"Winsorizacao: p{P_INFERIOR:.0%} a p{P_SUPERIOR:.0%} | cobertura minima: {COBERTURA_MINIMA:.0%}")
print(f"Correlacao maxima entre preditoras: {LIMIAR_CORRELACAO} | corte treino/teste: ate {ANO_CORTE_TREINO}")

Entrada: G:\Meu Drive\TCC\Damaris Rebeca\03_analitico
Saida: G:\Meu Drive\TCC\Damaris Rebeca\04_modelagem
Winsorizacao: p1% a p99% | cobertura minima: 60%
Correlacao maxima entre preditoras: 0.95 | corte treino/teste: ate 2021


In [5]:
def normalizar(s):
    s = unicodedata.normalize("NFKD", str(s) if pd.notna(s) else "")
    s = s.encode("ascii", "ignore").decode("ascii")
    return " ".join(s.upper().split())

def ler(nome, pasta=DIR_ANALITICO):
    caminho = Path(pasta) / f"{nome}.parquet"
    if not caminho.exists():
        print(f"AUSENTE: {caminho.name}")
        return None
    df = pd.read_parquet(caminho)
    print(f"{nome:<34}{len(df):>10,} linhas x {df.shape[1]:>4} colunas")
    return df

def salvar(df, nome, pasta=DIR_MODELAGEM):
    caminho = Path(pasta) / f"{nome}.parquet"
    df.to_parquet(caminho, index=False)
    print(f"{nome:<34}{len(df):>10,} linhas x {df.shape[1]:>4} colunas -> {caminho.name}")
    return caminho

def numericas(df, excluir=()):
    fora = set(excluir)
    return [c for c in df.select_dtypes(include=[np.number]).columns
            if c not in fora and not c.startswith("y_")]

def winsorizar(df, colunas, por=None, pi=P_INFERIOR, ps=P_SUPERIOR):
    saida = df.copy()
    relatorio = []
    grupos = [(None, saida)] if por is None else list(saida.groupby(por))
    for chave, grupo in grupos:
        idx = grupo.index
        for c in colunas:
            if c not in grupo.columns:
                continue
            s = pd.to_numeric(grupo[c], errors="coerce")
            if s.notna().sum() < 20:
                continue
            li, ls = s.quantile(pi), s.quantile(ps)
            if pd.isna(li) or pd.isna(ls) or li == ls:
                continue
            cortados = int(((s < li) | (s > ls)).sum())
            saida.loc[idx, c] = s.clip(li, ls)
            relatorio.append({"grupo": chave, "variavel": c, "cortados": cortados})
    return saida, pd.DataFrame(relatorio)

def cobertura(df, colunas):
    linhas = []
    for c in colunas:
        s = pd.to_numeric(df[c], errors="coerce")
        linhas.append({"variavel": c, "preenchidos": int(s.notna().sum()),
                       "cobertura": s.notna().mean(),
                       "zeros": float((s == 0).mean()),
                       "unicos": int(s.nunique())})
    return pd.DataFrame(linhas).sort_values("cobertura")

def filtrar_correlacao(df, colunas, limiar=LIMIAR_CORRELACAO):
    base = df[colunas].apply(pd.to_numeric, errors="coerce")
    corr = base.corr().abs()
    superior = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    remover, pares = [], []
    for c in superior.columns:
        altos = superior[c][superior[c] > limiar]
        for outra, valor in altos.items():
            if outra in remover or c in remover:
                continue
            perde = c if base[c].notna().mean() <= base[outra].notna().mean() else outra
            remover.append(perde)
            pares.append({"variavel_a": outra, "variavel_b": c,
                          "correlacao": round(float(valor), 4), "removida": perde})
    mantidas = [c for c in colunas if c not in remover]
    return mantidas, pd.DataFrame(pares)

## 2. Leitura e diagnóstico de entrada

In [7]:
print(f"{'arquivo':<34}{'linhas':>10}{'':>8}{'colunas':>4}\n")
a_bruta = ler("camada_a_cvm_painel")
b_bruta = ler("camada_b_bacen_painel")
c_bruta = ler("camada_c_rfb_coorte_mg")

print("\nsituacao dos rotulos na entrada:")
for nome, df, alvos in [("A", a_bruta, ["y_rj", "y_rj_h2", "y_rj_h3", "y_distress"]),
                        ("B", b_bruta, ["y_resolucao"]),
                        ("C", c_bruta, ["y_baixa"])]:
    if df is None:
        continue
    for alvo in alvos:
        if alvo in df.columns:
            n = int(df[alvo].sum())
            print(f"   camada {nome} | {alvo:<12} {n:>8,} positivos ({df[alvo].mean():.2%})")

arquivo                               linhas        colunas

camada_a_cvm_painel                   10,172 linhas x   75 colunas
camada_b_bacen_painel                 16,552 linhas x  128 colunas
camada_c_rfb_coorte_mg             1,415,279 linhas x   39 colunas

situacao dos rotulos na entrada:
   camada A | y_rj               48 positivos (0.47%)
   camada A | y_rj_h2            93 positivos (0.91%)
   camada A | y_rj_h3           136 positivos (1.34%)
   camada A | y_distress      2,386 positivos (23.46%)
   camada B | y_resolucao         0 positivos (0.00%)
   camada C | y_baixa       535,270 positivos (37.82%)


---
# 3. Camada A: companhias abertas

## 3.1 Correção da unidade temporal

O `ANO` do painel veio do nome do arquivo da CVM, não do exercício social. Empresas com exercício não coincidente com o ano civil ficam deslocadas, e o arquivo de 2026 traz exercícios ainda em curso. O ano passa a ser derivado de `DT_FIM_EXERC`, e exercícios com cobertura muito baixa são cortados por serem incompletos.

In [9]:
a = a_bruta.copy()
a["DT_FIM_EXERC"] = pd.to_datetime(a["DT_FIM_EXERC"], errors="coerce")
a["ano_exercicio"] = a["DT_FIM_EXERC"].dt.year
a["mes_fim_exercicio"] = a["DT_FIM_EXERC"].dt.month

divergentes = int((a["ano_exercicio"] != a["ANO"]).sum())
print(f"{divergentes:,} observacoes com ano do arquivo diferente do exercicio ({divergentes/len(a):.1%})")
print("\ncompanhias por exercicio:")
contagem = a.groupby("ano_exercicio")["CNPJ_CIA"].nunique()
print(contagem.to_string())

mediana = contagem.median()
anos_validos = contagem[contagem >= 0.3 * mediana].index.tolist()
print(f"\nexercicios mantidos: {min(anos_validos)} a {max(anos_validos)}")
print("descartados por cobertura insuficiente:",
      sorted(set(contagem.index) - set(anos_validos)) or "nenhum")

a = a[a["ano_exercicio"].isin(anos_validos)].copy()
a["ANO"] = a["ano_exercicio"].astype(int)
print(f"\n{len(a):,} observacoes apos o corte temporal")

0 observacoes com ano do arquivo diferente do exercicio (0.0%)

companhias por exercicio:
ano_exercicio
2011    679
2012    662
2013    649
2014    652
2015    633
2016    625
2017    632
2018    629
2019    670
2020    731
2021    765
2022    729
2023    732
2024    709
2025    666
2026      8

exercicios mantidos: 2011 a 2025
descartados por cobertura insuficiente: [2026]

10,164 observacoes apos o corte temporal


## 3.2 Deduplicação e recorte do universo

Uma companhia pode aparecer duas vezes no mesmo exercício quando publica demonstrativo consolidado e individual em arquivos de anos diferentes. A regra de desempate prefere o consolidado e, empatando, a data de encerramento mais recente.

O setor financeiro é retirado da camada A por decisão de desenho: liquidação de banco é o evento da camada B, e manter as mesmas entidades nas duas contaminaria a comparação entre elas.

In [14]:
antes = len(a)
a["prioridade"] = (a["TIPO_DEMO"] == "con").astype(int)
a = (a.sort_values(["CNPJ_CIA", "ANO", "prioridade", "DT_FIM_EXERC"])
       .drop_duplicates(subset=["CNPJ_CIA", "ANO"], keep="last")
       .drop(columns=["prioridade"]))
print(f"{antes - len(a)} duplicatas removidas na chave companhia e exercicio")

setor_norm = a["setor"].map(normalizar)
eh_financeira = setor_norm.str.contains(
    r"BANCO|ARRENDAMENTO|SEGURO|INTERMEDIACAO FINANCEIRA|SECURITIZ|CREDITO IMOBILIARIO|FINANCEIR",
    regex=True, na=False)
a["setor_financeiro"] = eh_financeira.astype("int8")
print(f"\n{int(eh_financeira.sum()):,} observacoes de setor financeiro ({eh_financeira.mean():.1%})")
print(a.loc[eh_financeira, "setor"].value_counts().head(6).to_string())

if EXCLUIR_FINANCEIRAS_A:
    a = a[a["setor_financeiro"] == 0].copy()
    print(f"\nexcluidas | restam {len(a):,} observacoes e {a['CNPJ_CIA'].nunique():,} companhias")

print("\npositivos apos os cortes:", {k: int(a[k].sum()) for k in ["y_rj", "y_rj_h2", "y_rj_h3", "y_distress"] if k in a.columns})

1 duplicatas removidas na chave companhia e exercicio

1,524 observacoes de setor financeiro (15.0%)
setor
Securitização de Recebíveis                   774
Bancos                                        425
Arrendamento Mercantil                         97
Emp. Adm. Part. - Intermediação Financeira     80
Emp. Adm. Part. - Crédito Imobiliário          49
Intermediação Financeira                       47

excluidas | restam 8,639 observacoes e 981 companhias

positivos apos os cortes: {'y_rj': 48, 'y_rj_h2': 93, 'y_rj_h3': 136, 'y_distress': 2141}


## 3.3 Sanidade contábil

Antes de winsorizar, quatro checagens que apanham erro de origem em vez de mascará-lo: ativo total não positivo, identidade patrimonial violada, receita negativa e razões impossíveis. As linhas com ativo total ausente ou não positivo saem, porque quase todo indicador depende dele.

In [16]:
diag = []
diag.append(("ativo_total ausente ou nao positivo", int((a["ativo_total"].isna() | (a["ativo_total"] <= 0)).sum())))
identidade = (a["ativo_total"] - (a["exigivel_total"] + a["patrimonio_liquido"])).abs()
tolerancia = 0.01 * a["ativo_total"].abs()
diag.append(("identidade patrimonial violada acima de 1%", int((identidade > tolerancia).sum())))
diag.append(("receita liquida negativa", int((a["receita_liquida"] < 0).sum())))
diag.append(("patrimonio liquido negativo", int((a["patrimonio_liquido"] < 0).sum())))

for nome, n in diag:
    print(f"{nome:<48}{n:>8,}  ({n/len(a):.2%})")

a = a[a["ativo_total"].notna() & (a["ativo_total"] > 0)].copy()
print(f"\n{len(a):,} observacoes apos remover ativo total invalido")

ativo_total ausente ou nao positivo                   28  (0.32%)
identidade patrimonial violada acima de 1%            34  (0.39%)
receita liquida negativa                              19  (0.22%)
patrimonio liquido negativo                        1,069  (12.37%)

8,611 observacoes apos remover ativo total invalido


## 3.4 Winsorização por exercício

Os extremos observados na entrega 2, como `giro_ativo` acima de quatro milhões, vêm de razões com denominador quase nulo. A winsorização é feita **dentro de cada exercício**, o que preserva a variação entre anos e evita que um ano atípico contamine o corte dos demais. Nenhuma observação é perdida: os valores são apenas limitados aos percentis 1 e 99.

In [18]:
IND_A = ["liquidez_corrente", "liquidez_seca", "liquidez_imediata", "liquidez_geral",
         "endividamento_geral", "endividamento_financeiro", "composicao_endividamento",
         "alavancagem", "pl_sobre_ativo", "pl_sobre_exigivel", "imobilizacao_recursos",
         "divida_liquida_ativo", "roa", "roe", "margem_liquida", "margem_bruta",
         "margem_operacional", "ebit_sobre_ativo", "lucros_acumulados_ativo",
         "retorno_antes_tributos", "giro_ativo", "ccl_sobre_ativo", "capital_giro_receita",
         "cobertura_juros", "despesa_financeira_receita", "crescimento_receita",
         "crescimento_ativo", "kanitz_fi", "altman_z2_emergentes"]
IND_A = [c for c in IND_A if c in a.columns]

antes_stats = a[IND_A].describe().T[["min", "50%", "max"]].rename(
    columns={"min": "min_antes", "50%": "mediana_antes", "max": "max_antes"})

a, rel_wins = winsorizar(a, IND_A, por="ANO")

depois_stats = a[IND_A].describe().T[["min", "50%", "max"]].rename(
    columns={"min": "min_depois", "50%": "mediana_depois", "max": "max_depois"})

comparacao = antes_stats.join(depois_stats)
print(f"{int(rel_wins['cortados'].sum()):,} valores limitados\n")
print(comparacao[["min_antes", "min_depois", "max_antes", "max_depois"]].to_string())

4,772 valores limitados

                               min_antes  min_depois      max_antes  max_depois
liquidez_corrente                 0.0000      0.0004     7,766.7000    533.7986
liquidez_seca                     0.0000      0.0004     7,766.7000    533.7986
liquidez_imediata                -0.0000      0.0000     5,548.4286    415.0750
liquidez_geral                    0.0000      0.0009   856,526.3333    533.6789
endividamento_geral               0.0000      0.0000   171,394.0000     59.0657
endividamento_financeiro         -0.0027      0.0000   146,897.0000     15.7516
composicao_endividamento          0.0000      0.0000         1.0000      1.0000
alavancagem                 -10,282.0000    -77.0408   918,778.0000     72.0041
pl_sobre_ativo             -171,393.0000    -58.0657         1.0000      1.0000
pl_sobre_exigivel                -1.0000     -0.9834 2,852,551.0000  1,189.5822
imobilizacao_recursos       -10,257.1667    -31.4293   316,411.8889     53.0797
divida_liquida_

## 3.5 Cobertura, multicolinearidade e conjunto final

Variáveis com cobertura abaixo do mínimo são descartadas: imputar mais de 40% de uma coluna é inventar dado. Em seguida sai o filtro de correlação, que é o passo que a metodologia do trabalho já previa e que evita alimentar os modelos com o mesmo sinal repetido.

In [20]:
cob_a = cobertura(a, IND_A)
print("cobertura das preditoras da camada A\n")
print(cob_a.to_string(index=False))

descartadas = cob_a[cob_a["cobertura"] < COBERTURA_MINIMA]["variavel"].tolist()
IND_A_OK = [c for c in IND_A if c not in descartadas]
print(f"\n{len(descartadas)} descartadas por cobertura:", descartadas or "nenhuma")

cobertura das preditoras da camada A

                  variavel  preenchidos  cobertura  zeros  unicos
       crescimento_receita         6897     0.8010 0.0003    6713
         crescimento_ativo         7633     0.8864 0.0035    7461
            margem_liquida         7646     0.8879 0.0012    7496
              margem_bruta         7666     0.8903 0.0003    7314
      capital_giro_receita         7666     0.8903 0.0003    7524
        margem_operacional         7666     0.8903 0.0002    7523
despesa_financeira_receita         7666     0.8903 0.0312    7325
           cobertura_juros         7984     0.9272 0.0006    7836
                 kanitz_fi         8496     0.9866 0.0000    8330
      altman_z2_emergentes         8512     0.9885 0.0000    8345
         liquidez_corrente         8514     0.9887 0.0000    8321
             liquidez_seca         8514     0.9887 0.0000    8322
         liquidez_imediata         8514     0.9887 0.0113    8277
         pl_sobre_exigivel         853

In [21]:
IND_A_FINAL, pares_a = filtrar_correlacao(a, IND_A_OK)
print(f"{len(IND_A_OK)} preditoras antes | {len(IND_A_FINAL)} depois do filtro de correlacao\n")
if len(pares_a):
    print(pares_a.to_string(index=False))
else:
    print("nenhum par acima do limiar")
print("\npreditoras mantidas:")
print(", ".join(IND_A_FINAL))

29 preditoras antes | 24 depois do filtro de correlacao

              variavel_a             variavel_b  correlacao             removida
       liquidez_corrente          liquidez_seca      0.9995        liquidez_seca
     endividamento_geral         pl_sobre_ativo      1.0000       pl_sobre_ativo
endividamento_financeiro   divida_liquida_ativo      0.9686 divida_liquida_ativo
                     roa retorno_antes_tributos      0.9844                  roa
       liquidez_corrente              kanitz_fi      0.9501            kanitz_fi

preditoras mantidas:
liquidez_corrente, liquidez_imediata, liquidez_geral, endividamento_geral, endividamento_financeiro, composicao_endividamento, alavancagem, pl_sobre_exigivel, imobilizacao_recursos, roe, margem_liquida, margem_bruta, margem_operacional, ebit_sobre_ativo, lucros_acumulados_ativo, retorno_antes_tributos, giro_ativo, ccl_sobre_ativo, capital_giro_receita, cobertura_juros, despesa_financeira_receita, crescimento_receita, crescimento_at

In [22]:
CHAVES_A = ["CNPJ_CIA", "DENOM_CIA", "ANO", "DT_FIM_EXERC", "TIPO_DEMO", "uf", "setor",
            "situacao", "setor_financeiro", "descartar_pos_evento", "fonte"]
ALVOS_A = [c for c in ["y_rj", "y_rj_h2", "y_rj_h3", "y_distress"] if c in a.columns]
EXTRAS_A = [c for c in ["ativo_total", "receita_liquida", "patrimonio_liquido",
                        "lucro_liquido", "pl_negativo", "prejuizo", "kanitz_faixa"] if c in a.columns]

camada_a = a[[c for c in CHAVES_A if c in a.columns] + EXTRAS_A + IND_A_FINAL + ALVOS_A].copy()
camada_a["log_ativo"] = np.log(camada_a["ativo_total"])

analise = camada_a[camada_a["descartar_pos_evento"] == 0]
print(f"{len(camada_a):,} observacoes | {len(analise):,} elegiveis para modelagem\n")
for alvo in ALVOS_A:
    n = int(analise[alvo].sum())
    print(f"   {alvo:<12}{n:>6} positivos  ({n/len(analise):.2%})")

salvar(camada_a, "camada_a_wrangled")

8,611 observacoes | 8,453 elegiveis para modelagem

   y_rj            48 positivos  (0.57%)
   y_rj_h2         93 positivos  (1.10%)
   y_rj_h3        136 positivos  (1.61%)
   y_distress    2005 positivos  (23.72%)
camada_a_wrangled                      8,611 linhas x   47 colunas -> camada_a_wrangled.parquet


WindowsPath('G:/Meu Drive/TCC/Damaris Rebeca/04_modelagem/camada_a_wrangled.parquet')

---
# 4. Camada B: instituições financeiras

## 4.1 Limpeza dos rótulos de coluna

O IF.data devolve o nome do indicador com a fórmula colada depois de uma quebra de linha, como `Índice de Basileia \n(m) = (e) / (i)`. Como a letra da fórmula muda quando o BACEN reorganiza o relatório, **a mesma variável vira colunas diferentes em anos diferentes**. É isso que produziu 98 colunas em 2015 e 27 em 2025, e é a razão de o painel empilhado ter 128 colunas cheias de vazio.

A correção é cortar tudo a partir da quebra de linha, remover o sufixo de fórmula e, quando duas colunas originais limparem para o mesmo rótulo, fundi-las preenchendo uma com a outra.

In [24]:
CHAVES_B = {"ano", "data_base", "tipo_instituicao", "cod_inst", "instituicao", "tcb",
            "consolidacao", "controle", "uf", "cidade", "segmento", "fonte",
            "y_resolucao", "descartar_pos_evento"}

def limpar_rotulo(c):
    s = str(c).replace("\r", " ")
    s = s.split("\n")[0]
    s = re.sub(r"\(\s*[a-z]\d?\s*\)\s*=.*$", "", s, flags=re.I)
    s = re.sub(r"\(\s*[a-z]\d?\s*\)\s*$", "", s, flags=re.I)
    return re.sub(r"\s+", " ", s).strip()

b = b_bruta.copy()
medidas = [c for c in b.columns if c not in CHAVES_B]

grupos = {}
for orig in medidas:
    grupos.setdefault(limpar_rotulo(orig), []).append(orig)

fundidos = {k: v for k, v in grupos.items() if len(v) > 1}
print(f"{len(medidas)} colunas de medida | {len(grupos)} rotulos apos limpeza | {len(fundidos)} fundidos\n")
for k, v in list(fundidos.items())[:12]:
    print(f"{k[:52]:<54}<- {len(v)} colunas")

114 colunas de medida | 105 rotulos apos limpeza | 9 fundidos

Resultado de Operações de Câmbio                      <- 2 colunas
Ativos Ponderados pelo Risco (RWA)                    <- 2 colunas
Exposição Total                                       <- 2 colunas
RWA para Risco de Mercado                             <- 2 colunas
Razão de Alavancagem                                  <- 2 colunas
Índice de Basileia                                    <- 2 colunas
Índice de Capital Nível I                             <- 2 colunas
Índice de Capital Principal                           <- 2 colunas
Índice de Imobilização                                <- 2 colunas


In [25]:
novo = b[[c for c in b.columns if c in CHAVES_B]].copy()
for limpo, origens in grupos.items():
    if not limpo:
        continue
    serie = pd.to_numeric(b[origens[0]], errors="coerce")
    for extra in origens[1:]:
        serie = serie.fillna(pd.to_numeric(b[extra], errors="coerce"))
    novo[limpo] = serie

b = novo
medidas = [c for c in b.columns if c not in CHAVES_B]
print(f"painel reconstruido: {len(b):,} linhas x {b.shape[1]} colunas ({len(medidas)} medidas)")

painel reconstruido: 16,552 linhas x 119 colunas (105 medidas)


## 4.2 Cobertura ao longo do tempo

Uma variável só é útil se existir na maior parte da janela. A matriz abaixo mostra em quantos exercícios cada medida aparece preenchida, e o corte mantém apenas as presentes em pelo menos `MIN_ANOS_COBERTURA_B` anos. É esse passo que resolve a série irregular.

In [27]:
anos_b = sorted(b["ano"].unique())
presenca = {}
for c in medidas:
    s = pd.to_numeric(b[c], errors="coerce")
    por_ano = s.groupby(b["ano"]).apply(lambda x: x.notna().mean())
    presenca[c] = {"anos_com_dado": int((por_ano > 0.2).sum()),
                   "cobertura_geral": float(s.notna().mean())}

pres = pd.DataFrame(presenca).T.sort_values(["anos_com_dado", "cobertura_geral"], ascending=False)
print(f"{len(anos_b)} exercicios: {anos_b}\n")
print(pres.head(30).to_string())

medidas_ok = pres[pres["anos_com_dado"] >= MIN_ANOS_COBERTURA_B].index.tolist()
print(f"\n{len(medidas_ok)} de {len(medidas)} medidas presentes em pelo menos {MIN_ANOS_COBERTURA_B} exercicios")

12 exercicios: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

                                                          anos_com_dado  cobertura_geral
prejuizo                                                        12.0000           1.0000
Aplicações Interfinanceiras de Liquidez                         11.0000           0.9187
Arrendamento Mercantil Líquido de Provisão                      11.0000           0.9187
Arrendamento Mercantil a Receber                                11.0000           0.9187
Ativo Total                                                     11.0000           0.9187
Ativo Total Ajustado                                            11.0000           0.9187
Captações                                                       11.0000           0.9187
Carteira de Crédito Classificada                                11.0000        

## 4.3 Indicadores reconciliados

Com os rótulos limpos, os indicadores são localizados por nome exato. A escala do Índice de Basileia é conferida: o IF.data publica a razão entre patrimônio de referência e ativos ponderados pelo risco, que fica entre zero e um, enquanto a literatura e a regulação falam em percentual. A conversão é aplicada quando a mediana indica que os valores estão em razão.

In [29]:
MAPA_B = {
    "ativo_total": ["Ativo Total"],
    "patrimonio_liquido": ["Patrimônio Líquido"],
    "lucro_liquido": ["Lucro Líquido"],
    "captacoes": ["Captações"],
    "carteira_credito": ["Carteira de Crédito Classificada"],
    "passivo_exigivel": ["Passivo Circulante e Exigível a Longo Prazo"],
    "basileia": ["Índice de Basileia"],
    "capital_principal": ["Índice de Capital Principal"],
    "patrimonio_referencia": ["Patrimônio de Referência para Comparação com o RWA"],
    "rwa": ["Ativos Ponderados pelo Risco (RWA)"],
}

encontrados = {}
for destino, nomes in MAPA_B.items():
    achada = None
    for n in nomes:
        alvo = normalizar(n)
        for c in medidas_ok:
            if normalizar(c) == alvo:
                achada = c
                break
        if achada:
            break
    encontrados[destino] = achada
    if achada:
        b[destino] = pd.to_numeric(b[achada], errors="coerce")

print(f"{'indicador':<24}{'coluna do IF.data'}")
for k, v in encontrados.items():
    print(f"{k:<24}{v if v else 'NAO ENCONTRADO'}")

for col in ["basileia", "capital_principal"]:
    if col in b.columns:
        mediana = b[col].median()
        if pd.notna(mediana) and mediana < 1.5:
            b[col] = b[col] * 100
            print(f"\n{col}: mediana era {mediana:.4f}, convertido de razao para percentual")

indicador               coluna do IF.data
ativo_total             Ativo Total
patrimonio_liquido      Patrimônio Líquido
lucro_liquido           NAO ENCONTRADO
captacoes               Captações
carteira_credito        Carteira de Crédito Classificada
passivo_exigivel        Passivo Circulante e Exigível a Longo Prazo
basileia                Índice de Basileia
capital_principal       Índice de Capital Principal
patrimonio_referencia   Patrimônio de Referência para Comparação com o RWA
rwa                     Ativos Ponderados pelo Risco (RWA)

basileia: mediana era 0.2730, convertido de razao para percentual

capital_principal: mediana era 0.2435, convertido de razao para percentual


In [30]:
b["roa"] = np.where(b["ativo_total"] > 0, b["lucro_liquido"] / b["ativo_total"], np.nan)
b["roe"] = np.where(b["patrimonio_liquido"] > 0, b["lucro_liquido"] / b["patrimonio_liquido"], np.nan)
b["pl_sobre_ativo"] = np.where(b["ativo_total"] > 0, b["patrimonio_liquido"] / b["ativo_total"], np.nan)
b["alavancagem"] = np.where(b["patrimonio_liquido"] > 0, b["ativo_total"] / b["patrimonio_liquido"], np.nan)
b["credito_sobre_ativo"] = np.where(b["ativo_total"] > 0, b["carteira_credito"] / b["ativo_total"], np.nan)
b["captacoes_sobre_ativo"] = np.where(b["ativo_total"] > 0, b["captacoes"] / b["ativo_total"], np.nan)
if encontrados.get("rwa"):
    b["rwa_sobre_ativo"] = np.where(b["ativo_total"] > 0, b["rwa"] / b["ativo_total"], np.nan)

b.loc[(b["pl_sobre_ativo"] < -2) | (b["pl_sobre_ativo"] > 1.5), "pl_sobre_ativo"] = np.nan
b.loc[(b["credito_sobre_ativo"] < 0) | (b["credito_sobre_ativo"] > 1.5), "credito_sobre_ativo"] = np.nan
b.loc[(b["captacoes_sobre_ativo"] < 0) | (b["captacoes_sobre_ativo"] > 2), "captacoes_sobre_ativo"] = np.nan
b.loc[(b["basileia"] < 0) | (b["basileia"] > 200), "basileia"] = np.nan

b = b.sort_values(["cod_inst", "ano"])
b["crescimento_ativo"] = b.groupby("cod_inst")["ativo_total"].pct_change()
b["prejuizo"] = (b["lucro_liquido"] < 0).astype("int8")
b["ativo_total_valido"] = (b["ativo_total"] > 0).astype("int8")

IND_B = [c for c in ["roa", "roe", "pl_sobre_ativo", "alavancagem", "basileia",
                     "capital_principal", "credito_sobre_ativo", "captacoes_sobre_ativo",
                     "rwa_sobre_ativo", "crescimento_ativo"] if c in b.columns]
print(b[IND_B].describe().T[["count", "min", "50%", "max"]].to_string())

                            count         min     50%             max
roa                    7,242.0000     -4.3317  0.0153          1.7217
roe                    7,218.0000 -3,129.8638  0.0586          4.2836
pl_sobre_ativo        15,199.0000     -1.8814  0.2254          1.0000
alavancagem           15,150.0000      1.0000  4.4244      7,574.3677
basileia              14,371.0000      0.0417 27.1035        199.9775
capital_principal      7,702.0000 -6,469.8760 24.3485    313,054.7704
credito_sobre_ativo   15,194.0000      0.0000  0.5536          1.4950
captacoes_sobre_ativo 15,206.0000      0.0000  0.4869          1.4702
rwa_sobre_ativo       13,425.0000      0.0000  0.7204 17,602,850.0000
crescimento_ativo     14,584.0000     -1.0000  0.1218 10,479,610.2300


In [31]:
b = b[b["ativo_total_valido"] == 1].copy()
b, rel_wins_b = winsorizar(b, IND_B, por="ano")
print(f"{int(rel_wins_b['cortados'].sum()):,} valores limitados\n")
print(b[IND_B].describe().T[["count", "min", "50%", "max"]].to_string())

2,186 valores limitados

                            count     min     50%      max
roa                    7,242.0000 -0.2914  0.0153   0.2299
roe                    7,218.0000 -0.9855  0.0586   0.4310
pl_sobre_ativo        15,199.0000  0.0267  0.2254   0.9931
alavancagem           15,150.0000  1.0069  4.4244  33.1911
basileia              13,090.0000  6.3857 27.3092 182.0809
capital_principal      7,262.0000 -4.5706 24.7076 887.3554
credito_sobre_ativo   15,194.0000  0.0000  0.5536   1.0296
captacoes_sobre_ativo 15,206.0000  0.0000  0.4869   0.9085
rwa_sobre_ativo       13,425.0000  0.0000  0.7204   5.5637
crescimento_ativo     13,299.0000 -0.5756  0.1444   6.2192


## 4.4 Rótulo de resolução

Na execução da entrega 2 a pasta `01_bruto/bcb_resolucao` estava vazia, e o rótulo saiu zerado. Sem variável resposta a camada B não modela nada.

A célula abaixo tenta três caminhos, em ordem. Primeiro, baixar do portal de dados abertos. Segundo, ler um arquivo manual que você mesma prepare. Terceiro, varrer o cadastro do IF.data em busca de instituições cujo nome carregue o carimbo de liquidação ou intervenção, que é o mesmo truque que funcionou na camada A com o cadastro da CVM.

In [33]:
DIR_RES = RAIZ / "01_bruto" / "bcb_resolucao"
DIR_RES.mkdir(parents=True, exist_ok=True)
CKAN = "https://dadosabertos.bcb.gov.br/api/3/action/package_show?id=instituicoes-submetidas-a-regimes-de-resolucao"

if not any(DIR_RES.glob("*")):
    try:
        import requests
        pacote = requests.get(CKAN, timeout=120).json()
        n = 0
        for r in pacote.get("result", {}).get("resources", []):
            fmt = (r.get("format") or "").upper()
            if fmt not in ("CSV", "XLSX", "XLS"):
                continue
            ext = {"CSV": ".csv", "XLSX": ".xlsx", "XLS": ".xls"}[fmt]
            nome = re.sub(r"[^A-Za-z0-9]+", "_", normalizar(r.get("name", "recurso")))[:60] + ext
            conteudo = requests.get(r["url"], timeout=300).content
            (DIR_RES / nome).write_bytes(conteudo)
            n += 1
        print(f"{n} recursos baixados do portal para {DIR_RES}")
    except Exception as e:
        print("download automatico falhou:", type(e).__name__, str(e)[:90])
        print("Baixe manualmente em https://dadosabertos.bcb.gov.br/dataset/instituicoes-submetidas-a-regimes-de-resolucao")
else:
    print(f"{len(list(DIR_RES.glob('*')))} arquivos ja presentes em {DIR_RES}")

1 arquivos ja presentes em G:\Meu Drive\TCC\Damaris Rebeca\01_bruto\bcb_resolucao


In [34]:
def carregar_resolucao():
    partes = []
    for p in sorted(DIR_RES.glob("*")):
        if p.suffix.lower() not in (".csv", ".xlsx", ".xls"):
            continue
        try:
            if p.suffix.lower() == ".csv":
                try:
                    t = pd.read_csv(p, sep=";", encoding="latin-1", dtype=str)
                except Exception:
                    t = pd.read_csv(p, sep=",", encoding="utf-8", dtype=str)
            else:
                t = pd.read_excel(p, dtype=str)
            t["recurso"] = p.name
            partes.append(t)
            print(f"{p.name:<52}{len(t):>6} linhas | {list(t.columns)[:5]}")
        except Exception as e:
            print(f"{p.name}: {type(e).__name__}")
    manual = DIR_INTER / "resolucao_manual.csv"
    if manual.exists():
        t = pd.read_csv(manual, dtype=str)
        t["recurso"] = "manual"
        partes.append(t)
        print(f"{'resolucao_manual.csv':<52}{len(t):>6} linhas")
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()

resolucao = carregar_resolucao()
print(f"\n{len(resolucao)} registros de resolucao carregados")

carimbo = b["instituicao"].map(normalizar).str.contains(
    r"EM LIQUIDACAO|EM INTERVENCAO|LIQUIDACAO EXTRAJUDICIAL|RAET", regex=True, na=False)
candidatos_nome = (b[carimbo].sort_values("ano")
                     .drop_duplicates(subset=["cod_inst"])[["cod_inst", "instituicao", "ano", "uf"]])
print(f"\n{len(candidatos_nome)} instituicoes com carimbo de regime no proprio nome do cadastro")
if len(candidatos_nome):
    print(candidatos_nome.head(15).to_string(index=False))


0 registros de resolucao carregados

0 instituicoes com carimbo de regime no proprio nome do cadastro


In [35]:
def achar_col(df, *chaves):
    for k in chaves:
        alvo = normalizar(k)
        for c in df.columns:
            if alvo in normalizar(c):
                return c
    return None

eventos_b = pd.DataFrame(columns=["cod_inst", "instituicao_evento", "ano_evento", "origem"])

if len(resolucao):
    col_nome = achar_col(resolucao, "instituicao", "nome", "razao social", "entidade")
    col_data = achar_col(resolucao, "data", "decreta", "inicio")
    col_ano = achar_col(resolucao, "ano")
    print("nome:", col_nome, "| data:", col_data, "| ano:", col_ano)

    if col_nome and (col_data or col_ano):
        ev = resolucao[[c for c in [col_nome, col_data, col_ano] if c]].copy()
        ev.columns = ["instituicao_evento"] + (["data"] if col_data else []) + (["ano"] if col_ano else [])
        if col_data:
            ev["ano_evento"] = pd.to_datetime(ev["data"], errors="coerce", dayfirst=True).dt.year
        if col_ano and "ano_evento" not in ev.columns:
            ev["ano_evento"] = pd.to_numeric(ev["ano"], errors="coerce")
        ev = ev.dropna(subset=["ano_evento"])
        ev["chave"] = ev["instituicao_evento"].map(normalizar)

        catalogo = b.drop_duplicates(subset=["cod_inst"])[["cod_inst", "instituicao", "uf"]].copy()
        catalogo["chave"] = catalogo["instituicao"].map(normalizar)
        from difflib import get_close_matches
        chaves = catalogo["chave"].tolist()
        mapa = dict(zip(catalogo["chave"], catalogo["cod_inst"]))

        linhas = []
        for _, e in ev.iterrows():
            aprox = get_close_matches(e["chave"], chaves, n=1, cutoff=0.75)
            linhas.append({"instituicao_evento": e["instituicao_evento"],
                           "ano_evento": int(e["ano_evento"]),
                           "sugestao_ifdata": aprox[0] if aprox else "",
                           "cod_inst": mapa.get(aprox[0]) if aprox else None,
                           "origem": "portal", "confirmado": "S" if aprox else ""})
        pareamento = pd.DataFrame(linhas)
        pareamento.to_csv(DIR_INTER / "pareamento_resolucao.csv", index=False, encoding="utf-8-sig")
        eventos_b = pareamento[pareamento["cod_inst"].notna()][
            ["cod_inst", "instituicao_evento", "ano_evento", "origem"]]
        print(f"\n{len(pareamento)} eventos | {len(eventos_b)} pareados automaticamente")
        print("confira e ajuste 02_intermediario/pareamento_resolucao.csv antes de considerar final")

if eventos_b.empty and len(candidatos_nome):
    eventos_b = candidatos_nome.rename(columns={"instituicao": "instituicao_evento", "ano": "ano_evento"})
    eventos_b["origem"] = "carimbo no nome"
    print(f"\nusando {len(eventos_b)} candidatos derivados do nome do cadastro")

print(f"\n{len(eventos_b)} eventos disponiveis para rotular a camada B")


0 eventos disponiveis para rotular a camada B


In [36]:
b["y_resolucao"] = 0
b["descartar_pos_evento"] = 0

for _, ev in eventos_b.iterrows():
    m = b["cod_inst"] == ev["cod_inst"]
    ano = int(ev["ano_evento"])
    b.loc[m & (b["ano"] == ano - 1), "y_resolucao"] = 1
    b.loc[m & (b["ano"] >= ano), "descartar_pos_evento"] = 1

elegivel_b = b[b["descartar_pos_evento"] == 0]
print(f"{len(b):,} observacoes | {len(elegivel_b):,} elegiveis")
print(f"positivos: {int(elegivel_b['y_resolucao'].sum())} ({elegivel_b['y_resolucao'].mean():.3%})")

if int(elegivel_b["y_resolucao"].sum()) < 30:
    print("\nATENCAO: menos de 30 positivos. A camada B ainda nao sustenta modelagem propria.")
    print("Confira o pareamento e considere ampliar a janela para trimestral na coleta.")

CHAVES_SAIDA_B = [c for c in ["ano", "data_base", "tipo_instituicao", "cod_inst", "instituicao",
                              "tcb", "consolidacao", "controle", "uf", "cidade", "segmento",
                              "descartar_pos_evento", "fonte"] if c in b.columns]
camada_b = b[CHAVES_SAIDA_B + ["ativo_total", "patrimonio_liquido", "lucro_liquido", "prejuizo"]
             + IND_B + ["y_resolucao"]].copy()
camada_b["log_ativo"] = np.log(camada_b["ativo_total"].clip(lower=1))
salvar(camada_b, "camada_b_wrangled")

15,207 observacoes | 15,207 elegiveis
positivos: 0 (0.000%)

ATENCAO: menos de 30 positivos. A camada B ainda nao sustenta modelagem propria.
Confira o pareamento e considere ampliar a janela para trimestral na coleta.
camada_b_wrangled                     15,207 linhas x   29 colunas -> camada_b_wrangled.parquet


WindowsPath('G:/Meu Drive/TCC/Damaris Rebeca/04_modelagem/camada_b_wrangled.parquet')

---
# 5. Camada C: coorte de Minas Gerais

## 5.1 O rótulo precisa ser desmembrado

A entrega 2 marcou 535 mil baixas como evento, mas a lista de motivos mostra o problema: 705 mil delas são "extinção por encerramento liquidação voluntária", que é o motivo padrão quando o empresário dá baixa a pedido. Isso não distingue quem fechou por falta de caixa de quem fechou por escolha. Junto vêm 13 mil óbitos de titular de MEI, que não é evento econômico, e 444 incorporações, que são reorganização societária e não morte da empresa.

Chamar tudo isso de insolvência seria impreciso e a banca cobraria. A saída correta é ser explícito sobre o que a base mede e produzir três rótulos:

| Rótulo | Definição | Para que serve |
|---|---|---|
| `y_mortalidade` | Baixa na janela por qualquer motivo, exceto incorporação e óbito do titular | Alvo principal, alinhado à literatura de sobrevivência de empresas |
| `y_baixa_involuntaria` | Inaptidão, registro cancelado, inexistência de fato | Aproximação de encerramento não escolhido |
| `y_insolvencia_formal` | Falência, liquidação judicial ou encerramento de liquidação | Insolvência em sentido estrito, raríssima nesse universo |

Com isso a camada C passa a ser honestamente descrita como **mortalidade empresarial**, que dialoga com a literatura do SEBRAE, e a insolvência stricto sensu fica nas camadas A e B.

In [38]:
c = c_bruta.copy()
c["motivo_norm"] = c["motivo_descricao"].map(normalizar)

def classificar_motivo(m):
    if not m:
        return "sem motivo"
    if "INCORPORACAO" in m or "FUSAO" in m or "CISAO" in m:
        return "reorganizacao"
    if "OBITO" in m:
        return "obito do titular"
    if "FALENCIA" in m or "LIQUIDACAO JUDICIAL" in m:
        return "insolvencia formal"
    if "ENCERRAMENTO DA LIQUIDACAO" in m:
        return "insolvencia formal"
    if "VOLUNTARIA" in m:
        return "encerramento voluntario"
    if "INAPT" in m or "CANCELADO" in m or "INEXISTENCIA" in m or "OMISS" in m:
        return "involuntario"
    return "outros"

c["classe_motivo"] = c["motivo_norm"].map(classificar_motivo)
print(c[c["baixada"] == 1]["classe_motivo"].value_counts().to_string())

classe_motivo
encerramento voluntario    705545
obito do titular            13399
involuntario                 1519
reorganizacao                 458
insolvencia formal            157
outros                         11


In [39]:
dentro = c["anos_ate_situacao"].between(0, JANELA_ANOS) if "anos_ate_situacao" in c.columns else False
if "anos_ate_situacao" not in c.columns:
    c["anos_ate_situacao"] = (c["dt_situacao"] - c["dt_abertura"]).dt.days / 365.25
    dentro = c["anos_ate_situacao"].between(0, JANELA_ANOS)

baixa = c["baixada"].eq(1) & dentro
c["y_mortalidade"] = (baixa & ~c["classe_motivo"].isin(["reorganizacao", "obito do titular"])).astype("int8")
c["y_baixa_involuntaria"] = (baixa & c["classe_motivo"].eq("involuntario")).astype("int8")
c["y_insolvencia_formal"] = (baixa & c["classe_motivo"].eq("insolvencia formal")).astype("int8")
c["y_encerramento_voluntario"] = (baixa & c["classe_motivo"].eq("encerramento voluntario")).astype("int8")

c["tempo_ate_evento_anos"] = np.where(c["y_mortalidade"] == 1, c["anos_ate_situacao"],
                                      c["idade_empresa_anos"].clip(upper=JANELA_ANOS))
c["evento_observado"] = c["y_mortalidade"]

for alvo in ["y_mortalidade", "y_baixa_involuntaria", "y_insolvencia_formal", "y_encerramento_voluntario"]:
    print(f"{alvo:<28}{int(c[alvo].sum()):>9,}  ({c[alvo].mean():.2%})")

print("\ntaxa de mortalidade por ano de abertura:")
print(c.groupby("ano_abertura")["y_mortalidade"].agg(["count", "mean"]).to_string())

y_mortalidade                 527,903  (37.30%)
y_baixa_involuntaria            1,512  (0.11%)
y_insolvencia_formal               55  (0.00%)
y_encerramento_voluntario     526,326  (37.19%)

taxa de mortalidade por ano de abertura:
               count   mean
ano_abertura               
2015          218569 0.3264
2016          306148 0.5000
2017          251987 0.3386
2018          294626 0.3401
2019          343949 0.3429


A taxa de 2016, bem acima dos demais anos, merece uma nota no texto: coincide com a onda de baixas de MEI daquele período. Não é erro de dado, é fenômeno real, e o ano de abertura entra como variável de controle justamente por isso.

## 5.2 Variáveis derivadas

A divisão CNAE de dois dígitos é granular demais para virar variável categórica em modelo de árvore. Ela é agregada na **seção CNAE**, a letra de A a U, que é a hierarquia oficial. O porte do município entra como proxy de mercado local, medido pelo número de empresas da coorte naquele município.

In [41]:
FAIXAS_SECAO = [
    ("A", 1, 3), ("B", 5, 9), ("C", 10, 33), ("D", 35, 35), ("E", 36, 39),
    ("F", 41, 43), ("G", 45, 47), ("H", 49, 53), ("I", 55, 56), ("J", 58, 63),
    ("K", 64, 66), ("L", 68, 68), ("M", 69, 75), ("N", 77, 82), ("O", 84, 84),
    ("P", 85, 85), ("Q", 86, 88), ("R", 90, 93), ("S", 94, 96), ("T", 97, 97),
    ("U", 99, 99),
]

def secao_cnae(divisao):
    try:
        d = int(divisao)
    except (TypeError, ValueError):
        return "desconhecida"
    for letra, ini, fim in FAIXAS_SECAO:
        if ini <= d <= fim:
            return letra
    return "desconhecida"

c["cnae_secao"] = c["cnae_divisao"].map(secao_cnae)

porte_mun = c.groupby("municipio_nome")["cnpj_basico"].transform("size")
c["empresas_no_municipio"] = porte_mun
c["log_empresas_municipio"] = np.log1p(porte_mun)
try:
    c["porte_municipio"] = pd.qcut(porte_mun.rank(method="first"), q=5,
                                   labels=["muito pequeno", "pequeno", "medio", "grande", "capital"])
except Exception:
    c["porte_municipio"] = pd.qcut(porte_mun.rank(method="first"), q=5,
                                   labels=False, duplicates="drop").astype("string")

c["capital_social"] = pd.to_numeric(c["capital_social"], errors="coerce")
c["capital_zero"] = (c["capital_social"].fillna(0) <= 0).astype("int8")
c["log_capital_social"] = np.log1p(c["capital_social"].clip(lower=0))
c["socio_unico"] = (c["qtd_socios"] <= 1).astype("int8")
c["tem_contato"] = ((c["tem_email"].astype(bool)) | (c["tem_telefone"].astype(bool))).astype("int8")
c["abertura_primeiro_semestre"] = (c["mes_abertura"] <= 6).astype("int8")

print(c["cnae_secao"].value_counts().head(12).to_string())
print("\nmortalidade por secao CNAE (secoes com mais de mil empresas):")
por_secao = c.groupby("cnae_secao").agg(empresas=("cnpj_basico", "size"),
                                        mortalidade=("y_mortalidade", "mean"))
print(por_secao[por_secao["empresas"] > 1000].sort_values("mortalidade", ascending=False).to_string())

cnae_secao
G    378213
S    224549
C    136990
I    133767
F    123248
H     91947
M     83903
N     78644
P     45234
Q     28546
T     22026
J     21718

mortalidade por secao CNAE (secoes com mais de mil empresas):
            empresas  mortalidade
cnae_secao                       
S             224549       0.5333
P              45234       0.4295
J              21718       0.4022
M              83903       0.3784
I             133767       0.3743
G             378213       0.3583
R              16238       0.3430
N              78644       0.3311
H              91947       0.3309
C             136990       0.3243
E               3875       0.3177
T              22026       0.3156
A               7003       0.3010
F             123248       0.2863
Q              28546       0.2692
B               1062       0.1798
K               8866       0.1644
L               7798       0.1467
O               1174       0.0128


In [42]:
IND_C = ["log_capital_social", "capital_zero", "qtd_socios", "qtd_socios_pf", "socio_unico",
         "faixa_etaria_min", "faixa_etaria_max", "optante_simples", "optante_mei",
         "excluido_simples", "porte_num", "tem_email", "tem_telefone", "tem_contato",
         "tem_cnae_secundario", "abertura_primeiro_semestre", "mes_abertura",
         "log_empresas_municipio"]
IND_C = [x for x in IND_C if x in c.columns]
for x in IND_C:
    c[x] = pd.to_numeric(c[x], errors="coerce")

cob_c = cobertura(c, IND_C)
print(cob_c.to_string(index=False))

descarta_c = cob_c[cob_c["cobertura"] < COBERTURA_MINIMA]["variavel"].tolist()
IND_C_OK = [x for x in IND_C if x not in descarta_c]
print("\ndescartadas por cobertura:", descarta_c or "nenhuma")

IND_C_FINAL, pares_c = filtrar_correlacao(c, IND_C_OK)
print(f"\n{len(IND_C_OK)} preditoras antes | {len(IND_C_FINAL)} depois do filtro de correlacao")
if len(pares_c):
    print(pares_c.to_string(index=False))

                  variavel  preenchidos  cobertura  zeros  unicos
          faixa_etaria_max       220684     0.1559 0.0000       9
          faixa_etaria_min       220684     0.1559 0.0421      10
        log_capital_social      1415279     1.0000 0.0772   11112
              capital_zero      1415279     1.0000 0.9228       2
             qtd_socios_pf      1415279     1.0000 0.8441      51
                qtd_socios      1415279     1.0000 0.8441      55
               socio_unico      1415279     1.0000 0.0704       2
           optante_simples      1415279     1.0000 0.6279       2
               optante_mei      1415279     1.0000 0.7196       2
          excluido_simples      1415279     1.0000 0.4794       2
                 porte_num      1415279     1.0000 0.0000       3
                 tem_email      1415279     1.0000 0.1763       2
              tem_telefone      1415279     1.0000 0.0000       2
               tem_contato      1415279     1.0000 0.0000       2
       tem

In [43]:
CHAVES_C = ["cnpj_basico", "razao_social", "uf", "municipio_nome", "porte_municipio",
            "cnae_fiscal_principal", "cnae_divisao", "cnae_secao", "natureza_descricao",
            "ano_abertura", "dt_abertura", "situacao_cadastral", "classe_motivo",
            "motivo_descricao", "tempo_ate_evento_anos", "evento_observado", "fonte"]
ALVOS_C = ["y_mortalidade", "y_baixa_involuntaria", "y_insolvencia_formal", "y_encerramento_voluntario"]

camada_c = c[[x for x in CHAVES_C if x in c.columns] + IND_C_FINAL + ALVOS_C].copy()
salvar(camada_c, "camada_c_wrangled")

if len(camada_c) > AMOSTRA_C:
    fracao = AMOSTRA_C / len(camada_c)
    amostra = camada_c.loc[
        camada_c.groupby(["ano_abertura", "y_mortalidade"]).sample(frac=fracao, random_state=42).index
    ].copy()
    print(f"\namostra estratificada por ano de abertura e desfecho: {len(amostra):,} de {len(camada_c):,}")
    print("taxa na amostra: {:.2%} | na base completa: {:.2%}".format(
        amostra["y_mortalidade"].mean(), camada_c["y_mortalidade"].mean()))
    salvar(amostra, "camada_c_wrangled_amostra")
else:
    amostra = camada_c

camada_c_wrangled                  1,415,279 linhas x   36 colunas -> camada_c_wrangled.parquet

amostra estratificada por ano de abertura e desfecho: 400,001 de 1,415,279
taxa na amostra: 37.30% | na base completa: 37.30%
camada_c_wrangled_amostra            400,001 linhas x   36 colunas -> camada_c_wrangled_amostra.parquet


---
# 6. Recorte de Minas Gerais e partição

## 6.1 O recorte

A camada C já é mineira por construção. As camadas A e B são recortadas aqui, e ambas as versões ficam gravadas.

In [45]:
def recortar(df, nome, coluna="uf"):
    if df is None or df.empty:
        return None
    mg = df[df[coluna].fillna("").str.upper().str.strip() == UF_RECORTE].copy()
    salvar(mg, f"{nome}_{UF_RECORTE.lower()}")
    return mg

camada_a_mg = recortar(camada_a, "camada_a_wrangled")
camada_b_mg = recortar(camada_b, "camada_b_wrangled")

camada_a_wrangled_mg                     698 linhas x   47 colunas -> camada_a_wrangled_mg.parquet
camada_b_wrangled_mg                   2,286 linhas x   29 colunas -> camada_b_wrangled_mg.parquet


## 6.2 Partição temporal

A divisão treino e teste é **temporal**, não aleatória. Sortear observações de uma série no tempo vaza informação do futuro para o treino e infla artificialmente o desempenho, o que é uma das críticas mais comuns em banca. As camadas A e B usam o exercício como eixo. A camada C é uma coorte, então o eixo é o ano de abertura.

O K-Fold estratificado continua disponível na modelagem, mas como análise complementar de estabilidade, não como validação principal.

In [47]:
def particionar_temporal(df, coluna_tempo, corte, nome):
    if df is None or df.empty:
        return None, None
    treino = df[df[coluna_tempo] <= corte].copy()
    teste = df[df[coluna_tempo] > corte].copy()
    treino["particao"] = "treino"
    teste["particao"] = "teste"
    print(f"{nome:<28}treino {len(treino):>9,} ({coluna_tempo} ate {corte})   teste {len(teste):>9,}")
    return treino, teste

print("particoes\n")
a_treino, a_teste = particionar_temporal(
    camada_a[camada_a["descartar_pos_evento"] == 0], "ANO", ANO_CORTE_TREINO, "A CVM nacional")
b_treino, b_teste = particionar_temporal(
    camada_b[camada_b["descartar_pos_evento"] == 0], "ano", ANO_CORTE_TREINO, "B BACEN nacional")

CORTE_C = 2017
c_treino, c_teste = particionar_temporal(camada_c, "ano_abertura", CORTE_C, "C RFB MG")

print("\npositivos por particao\n")
for nome, tr, te, alvo in [("A y_rj", a_treino, a_teste, "y_rj"),
                           ("A y_distress", a_treino, a_teste, "y_distress"),
                           ("B y_resolucao", b_treino, b_teste, "y_resolucao"),
                           ("C y_mortalidade", c_treino, c_teste, "y_mortalidade")]:
    if tr is None or alvo not in tr.columns:
        continue
    print(f"{nome:<20}treino {int(tr[alvo].sum()):>7,} de {len(tr):>9,} ({tr[alvo].mean():.2%})"
          f"   teste {int(te[alvo].sum()):>7,} de {len(te):>9,} ({te[alvo].mean():.2%})")

particoes

A CVM nacional              treino     5,957 (ANO ate 2021)   teste     2,496
B BACEN nacional            treino    11,156 (ano ate 2021)   teste     4,051
C RFB MG                    treino   776,704 (ano_abertura ate 2017)   teste   638,575

positivos por particao

A y_rj              treino      32 de     5,957 (0.54%)   teste      16 de     2,496 (0.64%)
A y_distress        treino   1,464 de     5,957 (24.58%)   teste     541 de     2,496 (21.67%)
B y_resolucao       treino       0 de    11,156 (0.00%)   teste       0 de     4,051 (0.00%)
C y_mortalidade     treino 309,753 de   776,704 (39.88%)   teste 218,150 de   638,575 (34.16%)


In [48]:
for df, nome in [(a_treino, "camada_a_treino"), (a_teste, "camada_a_teste"),
                 (b_treino, "camada_b_treino"), (b_teste, "camada_b_teste"),
                 (c_treino, "camada_c_treino"), (c_teste, "camada_c_teste")]:
    if df is not None and len(df):
        salvar(df, nome)

camada_a_treino                        5,957 linhas x   48 colunas -> camada_a_treino.parquet
camada_a_teste                         2,496 linhas x   48 colunas -> camada_a_teste.parquet
camada_b_treino                       11,156 linhas x   30 colunas -> camada_b_treino.parquet
camada_b_teste                         4,051 linhas x   30 colunas -> camada_b_teste.parquet
camada_c_treino                      776,704 linhas x   37 colunas -> camada_c_treino.parquet
camada_c_teste                       638,575 linhas x   37 colunas -> camada_c_teste.parquet


## 6.3 Dicionário de dados

Documenta cada variável de cada camada com tipo, cobertura e papel. É anexo obrigatório de um trabalho quantitativo sério e evita que a banca precise adivinhar o que é cada coluna.

In [50]:
def descrever(df, camada, alvos, chaves):
    linhas = []
    for col in df.columns:
        s = df[col]
        if col in alvos:
            papel = "variavel resposta"
        elif col in chaves:
            papel = "identificacao"
        elif pd.api.types.is_numeric_dtype(s):
            papel = "preditora numerica"
        else:
            papel = "preditora categorica"
        linhas.append({
            "camada": camada, "variavel": col, "tipo": str(s.dtype), "papel": papel,
            "cobertura": round(float(s.notna().mean()), 4),
            "unicos": int(s.nunique(dropna=True)),
            "minimo": float(pd.to_numeric(s, errors="coerce").min()) if pd.api.types.is_numeric_dtype(s) else None,
            "maximo": float(pd.to_numeric(s, errors="coerce").max()) if pd.api.types.is_numeric_dtype(s) else None,
        })
    return pd.DataFrame(linhas)

dicionario = pd.concat([
    descrever(camada_a, "A CVM", ALVOS_A, CHAVES_A),
    descrever(camada_b, "B BACEN", ["y_resolucao"], CHAVES_SAIDA_B),
    descrever(camada_c, "C RFB", ALVOS_C, CHAVES_C),
], ignore_index=True)

dicionario.to_csv(DIR_MODELAGEM / "dicionario_de_dados.csv", index=False, encoding="utf-8-sig")
print(f"{len(dicionario)} variaveis documentadas\n")
print(dicionario.groupby(["camada", "papel"]).size().to_string())

112 variaveis documentadas

camada   papel               
A CVM    identificacao           11
         preditora categorica     1
         preditora numerica      31
         variavel resposta        4
B BACEN  identificacao           13
         preditora numerica      15
         variavel resposta        1
C RFB    identificacao           17
         preditora numerica      15
         variavel resposta        4


---
# 7. Verificação final

In [52]:
def verificar(nome, df, chave, alvo, preditoras):
    if df is None or df.empty:
        return {"base": nome, "linhas": 0, "problemas": ["base vazia"]}
    p = []
    if alvo not in df.columns:
        p.append("rotulo ausente")
    else:
        pos = int(df[alvo].sum())
        if pos == 0:
            p.append("nenhum caso positivo")
        elif pos < 30:
            p.append(f"apenas {pos} positivos, insuficiente para modelagem propria")
    faltam = [k for k in chave if k not in df.columns]
    if faltam:
        p.append(f"chave incompleta: {faltam}")
    else:
        dup = int(df.duplicated(subset=chave).sum())
        if dup:
            p.append(f"{dup} duplicatas na chave {chave}")
    for col in preditoras:
        if col not in df.columns:
            continue
        s = pd.to_numeric(df[col], errors="coerce")
        if s.notna().sum() == 0:
            p.append(f"{col} vazio")
        elif s.isna().mean() > (1 - COBERTURA_MINIMA):
            p.append(f"{col} com {s.isna().mean():.0%} de ausentes")
        elif not bool(np.isfinite(s.dropna()).all()):
            p.append(f"{col} contem infinito")
        elif s.abs().max() > 1e6:
            p.append(f"{col} ainda com valor extremo ({s.abs().max():,.0f})")
    return {"base": nome, "linhas": len(df), "problemas": p or ["ok"]}

checagens = [
    verificar("A CVM nacional, y_rj", camada_a[camada_a["descartar_pos_evento"] == 0],
              ["CNPJ_CIA", "ANO"], "y_rj", IND_A_FINAL),
    verificar("A CVM nacional, y_distress", camada_a[camada_a["descartar_pos_evento"] == 0],
              ["CNPJ_CIA", "ANO"], "y_distress", IND_A_FINAL),
    verificar(f"A CVM {UF_RECORTE}", camada_a_mg, ["CNPJ_CIA", "ANO"], "y_rj", IND_A_FINAL),
    verificar("B BACEN nacional", camada_b[camada_b["descartar_pos_evento"] == 0],
              ["cod_inst", "ano"], "y_resolucao", IND_B),
    verificar(f"C RFB {UF_RECORTE}", camada_c, ["cnpj_basico"], "y_mortalidade", IND_C_FINAL),
]

for d in checagens:
    print(f"\n{d['base']} ({d['linhas']:,} linhas)")
    for x in d["problemas"]:
        print("   -", x)


A CVM nacional, y_rj (8,453 linhas)
   - crescimento_receita contem infinito

A CVM nacional, y_distress (8,453 linhas)
   - crescimento_receita contem infinito

A CVM MG (698 linhas)
   - apenas 3 positivos, insuficiente para modelagem propria
   - crescimento_receita contem infinito

B BACEN nacional (15,207 linhas)
   - nenhum caso positivo
   - roa com 52% de ausentes
   - roe com 53% de ausentes
   - capital_principal com 52% de ausentes

C RFB MG (1,415,279 linhas)
   - ok


In [53]:
elegivel_a = camada_a[camada_a["descartar_pos_evento"] == 0]
if elegivel_a["y_rj"].sum() > 0:
    med = elegivel_a.groupby("y_rj")[IND_A_FINAL].median().T
    med.columns = ["saudaveis", "pre_insolvencia"]
    med["variacao"] = (med["pre_insolvencia"] - med["saudaveis"]) / med["saudaveis"].abs()
    print("Camada A, medianas apos tratamento\n")
    print(med.sort_values("variacao").to_string())

Camada A, medianas apos tratamento

                            saudaveis  pre_insolvencia  variacao
lucros_acumulados_ativo        0.0000          -0.5309      -inf
margem_liquida                 0.0553          -0.6760  -13.2236
capital_giro_receita           0.1353          -0.6187   -5.5745
retorno_antes_tributos         0.0305          -0.1346   -5.4082
crescimento_ativo              0.0572          -0.1475   -3.5788
ccl_sobre_ativo                0.0703          -0.1497   -3.1299
crescimento_receita            0.0904          -0.1358   -2.5023
ebit_sobre_ativo               0.0607          -0.0891   -2.4686
margem_operacional             0.1309          -0.1441   -2.1009
cobertura_juros                1.1756          -0.4376   -1.3723
altman_z2_emergentes           4.9098          -0.1300   -1.0265
imobilizacao_recursos          1.4694           0.0000   -1.0000
alavancagem                    1.2936           0.0050   -0.9962
liquidez_imediata              0.4553           0.0368

In [54]:
manifesto = {
    "gerado_em": datetime.now().isoformat(timespec="seconds"),
    "parametros": {
        "uf_recorte": UF_RECORTE,
        "winsorizacao": [P_INFERIOR, P_SUPERIOR],
        "cobertura_minima": COBERTURA_MINIMA,
        "limiar_correlacao": LIMIAR_CORRELACAO,
        "min_anos_cobertura_b": MIN_ANOS_COBERTURA_B,
        "corte_treino_ab": ANO_CORTE_TREINO,
        "corte_treino_c": CORTE_C,
        "excluir_financeiras_a": EXCLUIR_FINANCEIRAS_A,
    },
    "camada_a": {
        "observacoes": int(len(camada_a)),
        "companhias": int(camada_a["CNPJ_CIA"].nunique()),
        "exercicios": [int(camada_a["ANO"].min()), int(camada_a["ANO"].max())],
        "preditoras": IND_A_FINAL,
        "positivos": {k: int(elegivel_a[k].sum()) for k in ALVOS_A},
        "observacoes_mg": int(len(camada_a_mg)) if camada_a_mg is not None else 0,
    },
    "camada_b": {
        "observacoes": int(len(camada_b)),
        "instituicoes": int(camada_b["cod_inst"].nunique()),
        "preditoras": IND_B,
        "positivos": int(camada_b["y_resolucao"].sum()),
        "observacoes_mg": int(len(camada_b_mg)) if camada_b_mg is not None else 0,
    },
    "camada_c": {
        "empresas": int(len(camada_c)),
        "preditoras": IND_C_FINAL,
        "positivos": {k: int(camada_c[k].sum()) for k in ALVOS_C},
    },
    "arquivos": sorted(p.name for p in DIR_MODELAGEM.glob("*")),
}

(DIR_MODELAGEM / "manifesto_wrangling.json").write_text(
    json.dumps(manifesto, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifesto, ensure_ascii=False, indent=2))

{
  "gerado_em": "2026-09-09T23:08:53",
  "parametros": {
    "uf_recorte": "MG",
    "winsorizacao": [
      0.01,
      0.99
    ],
    "cobertura_minima": 0.6,
    "limiar_correlacao": 0.95,
    "min_anos_cobertura_b": 8,
    "corte_treino_ab": 2021,
    "corte_treino_c": 2017,
    "excluir_financeiras_a": true
  },
  "camada_a": {
    "observacoes": 8611,
    "companhias": 980,
    "exercicios": [
      2011,
      2025
    ],
    "preditoras": [
      "liquidez_corrente",
      "liquidez_imediata",
      "liquidez_geral",
      "endividamento_geral",
      "endividamento_financeiro",
      "composicao_endividamento",
      "alavancagem",
      "pl_sobre_exigivel",
      "imobilizacao_recursos",
      "roe",
      "margem_liquida",
      "margem_bruta",
      "margem_operacional",
      "ebit_sobre_ativo",
      "lucros_acumulados_ativo",
      "retorno_antes_tributos",
      "giro_ativo",
      "ccl_sobre_ativo",
      "capital_giro_receita",
      "cobertura_juros",
      "despes

---
## O que sai deste notebook

Em `04_modelagem`:

| Arquivo | Conteúdo |
|---|---|
| `camada_a_wrangled` e `_mg` | Companhias abertas, painel tratado |
| `camada_b_wrangled` e `_mg` | Instituições financeiras, painel tratado |
| `camada_c_wrangled` e `_amostra` | Coorte mineira completa e amostra estratificada |
| `camada_*_treino` e `_teste` | Partições temporais prontas |
| `dicionario_de_dados.csv` | Todas as variáveis descritas |
| `manifesto_wrangling.json` | Parâmetros e contagens da execução |

## Duas pendências que dependem de decisão sua

**O rótulo da camada B.** Se a contagem de positivos continuar abaixo de trinta depois deste notebook, a camada B não sustenta modelo próprio. Há dois caminhos: ampliar a coleta para frequência trimestral, o que multiplica por quatro as observações e capta eventos que o corte anual perde, ou rebaixar a camada B a análise descritiva comparativa. Não há terceira via com esse número de eventos.

**A camada A em Minas.** Com pouquíssimos eventos mineiros entre companhias abertas, treinar só com MG não é viável. O desenho defensável é treinar no nacional com `uf` entre as variáveis e apresentar Minas como recorte descritivo dentro do modelo nacional. A camada C é onde a delimitação mineira sustenta modelagem por conta própria.

A modelagem, com Regressão Logística, Random Forest, XGBoost, validação temporal e SHAP, entra na entrega 4.

In [86]:
alvo = ["Ativo Total", "Patrimônio Líquido", "Lucro Líquido",
        "Índice de Basileia", "Índice de Capital Principal"]
presenca = {}
for nome in alvo:
    col = next((x for x in camada_b.columns if normalizar(x) == normalizar(nome)), None)
    if col is None:
        for destino, origem in encontrados.items():
            if origem and normalizar(origem) == normalizar(nome):
                col = destino
    if col and col in camada_b.columns:
        s = pd.to_numeric(camada_b[col], errors="coerce")
        presenca[nome] = s.groupby(camada_b["ano"]).apply(lambda x: round(x.notna().mean(), 3))

print(pd.DataFrame(presenca).to_string())
print("\ninstituicoes por tipo e ano:")
print(camada_b.groupby(["ano", "tipo_instituicao"])["cod_inst"].nunique().unstack(fill_value=0).to_string())

      Ativo Total  Patrimônio Líquido  Índice de Basileia  Índice de Capital Principal
ano                                                                                   
2014       1.0000              1.0000              0.0000                       0.0000
2015       1.0000              1.0000              0.9590                       0.9730
2016       1.0000              1.0000              0.9600                       0.9750
2017       1.0000              1.0000              0.9600                       0.9710
2018       1.0000              1.0000              0.9560                       0.3230
2019       1.0000              1.0000              0.9790                       0.3280
2020       1.0000              1.0000              0.9630                       0.3340
2021       1.0000              1.0000              0.9570                       0.3360
2022       1.0000              1.0000              0.9260                       0.3220
2023       1.0000              1.0000      

In [88]:
candidatos = [c for c in b.columns
              if any(t in normalizar(c) for t in ["LUCRO", "RESULTADO", "PREJUIZO"])]
print(f"{len(candidatos)} colunas candidatas\n")
for c in candidatos:
    s = pd.to_numeric(b[c], errors="coerce")
    anos_ok = s.groupby(b["ano"]).apply(lambda x: x.notna().mean())
    print(f"{c[:56]:<58}{s.notna().mean():.1%} geral | anos com dado: {int((anos_ok > 0.2).sum())}")

13 colunas candidatas

Lucro Líquido                                             47.6% geral | anos com dado: 5
Participação nos Lucros                                   100.0% geral | anos com dado: 11
Passivo Circulante e Exigível a Longo Prazo e Resultados  100.0% geral | anos com dado: 11
Resultado Não Operacional                                 100.0% geral | anos com dado: 11
Resultado Operacional                                     100.0% geral | anos com dado: 11
Resultado antes da Tributação, Lucro e Participação       100.0% geral | anos com dado: 11
Resultado de Intermediação Financeira                     100.0% geral | anos com dado: 11
Resultado de Operações de Câmbio                          100.0% geral | anos com dado: 11
Resultado de Participações                                100.0% geral | anos com dado: 11
Resultado de Provisão para Créditos de Difícil Liquidaçã  100.0% geral | anos com dado: 11
Resultados de Exercícios Futuros                          82.2% geral